# Scala Basics for Data Engineers


*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is Scala and why do data engineers learn it?

Scala is a statically typed JVM language that blends object-oriented and functional programming. Spark is written in Scala, the official Kafka clients are Java, and Flink exposes a JVM DataStream API, so Scala sits naturally next to all three.

```text
Kafka (Java clients) -> Spark/Flink JVM job -> lakehouse tables -> BI / ML
```

Data engineers learn Scala because:

+ Spark's API and internals are Scala-first, so source code and error messages are easier to read
+ Typed `Dataset[T]` code catches schema mistakes at compile time instead of at runtime
+ JVM-native logic avoids Python UDF serialization overhead for CPU-bound transforms
+ One library can be shared by Spark jobs, Flink jobs, and microservices

Real-world example: a team keeps revenue logic in one Scala library, and both a Spark batch job and a Flink streaming job call the same `OrderTransforms` object, so batch and streaming cannot silently diverge.


Scala source compiles to JVM bytecode, so a Scala job runs in the same process as the Spark and Kafka JVM clients. That is why a Spark stack trace shows Scala class names: the driver and executors are JVMs running compiled Scala. Types are checked when you build, and the build fails before a job is ever submitted.



Worked example: one transforms object is compiled once and used two ways.

```scala
// src/main/scala/com/acme/OrderTransforms.scala
package com.acme

final case class Order(orderId: Long, amount: BigDecimal, status: String)

object OrderTransforms {
  def paidTotal(orders: Seq[Order]): BigDecimal =
    orders.filter(_.status == "PAID").map(_.amount).sum
}
```

The Spark job calls `OrderTransforms.paidTotal` on a collected `Seq`, and the Flink job calls the same method on a window, so both share one compiled artifact and one review.

Common mistake: treating Scala as a scripting language and skipping the build. The value of Scala is the compiler, so run `sbt compile` early instead of discovering type errors at submit time.


## How do vals and vars differ, and why does immutability matter?

`val` binds a name once; `var` can be reassigned. Default to `val` and use `var` only for tight local accumulation. `val` makes the reference immutable, not the object, so a `val` can still point at a mutable collection.

```scala
val rawPath = "s3://acme-raw/orders/"   // immutable binding
var attempt = 0                          // reassignable
```


This matters in Spark and Flink because closures are serialized and shipped to executors or task managers. A captured mutable variable is copied, so updates on executors are invisible on the driver. Immutable values remove that failure mode and are safe to share across threads and Futures.


A `val` compiles to a final local or field, so reassigning it is a compile error rather than a runtime surprise. A `var` compiles to a mutable slot. Immutability also makes a value safe to share, because nothing can change underneath you.

```scala
val threshold = BigDecimal("1000")   // final
var retries = 0                      // mutable slot
retries += 1                         // fine

// threshold = BigDecimal("2000")    // error: reassignment to val
```

Expected: `retries` is `1`, and the commented line does not compile.



Common mistake: reading `val` as "the object cannot change". The binding cannot point elsewhere, but the object can still be mutable.

```scala
val buffer = scala.collection.mutable.ArrayBuffer(1, 2, 3)
buffer += 4                // allowed: the binding is fixed, the buffer is not
// buffer = ArrayBuffer()  // error: reassignment to val
```

In Spark, prefer `val` plus immutable collections. If a closure captures the buffer above, executors mutate their own serialized copy and the driver never sees the change.


## What are Scala basic types and how does type inference work?

Scala has the JVM primitives (`Int`, `Long`, `Double`, `Boolean`, `Char`) plus `String`, `BigInt`, and `BigDecimal`. Everything is an object at the API level, with `Any` at the top, `AnyVal`/`AnyRef` below it, and `Nothing` at the bottom. The compiler infers types, so annotate when widening or when you want a stable contract.

```scala
val rows = 1_000_000L             // Long
val ratio = 0.42                  // Double
val ids = Vector(1L, 2L, 3L)      // Vector[Long]
val widened: Long = rows + 1      // explicit widening
val total: BigDecimal = BigDecimal("19.99") * 3
```

Two habits help in interviews:

+ Annotate public return types and dataset schemas; inference is for locals
+ `Nothing` is a subtype of everything, which is why `if (x) 1 else throw new RuntimeException` still infers `Int`

For money prefer `BigDecimal` over `Double`: Spark maps `decimal(p, s)` to `BigDecimal`, and floating-point rounding in revenue numbers is a classic production incident.


Worked example of inference, widening, and overflow:

```scala
val id = 7                 // Int
val idLong: Long = id      // Int widens to Long at the assignment
val count = 2_000_000      // Int, about 2.1 billion max
val big = count * 2000L    // Long arithmetic because one side is Long

val a = 1 / 2              // Int division: 0, not 0.5
val b = 1.0 / 2            // Double division: 0.5
```

Expected: `idLong` is `7L`, `big` is `4000000000L`, `a` is `0`, and `b` is `0.5`.



Common mistake: summing money in `Int` or comparing `Double` revenue for equality. `Int` wraps silently on overflow, and `Double` cannot represent `0.1` exactly.

```scala
val cents = List(Int.MaxValue, 1)
val overflow = cents.sum          // wraps to a negative number

val price = BigDecimal("0.1") + BigDecimal("0.2")   // exactly 0.3
val approx = 0.1 + 0.2                             // 0.30000000000000004
```

Use `Long` for counts and `BigDecimal` for money, and compare the decimal value rather than the `Double` approximation.


## How does string interpolation work in Scala?

Three built-in interpolators: `s` substitutes values, `f` formats them, and `raw` disables escape processing. Interpolation accepts expressions in braces, so paths and log lines read cleanly.

```scala
val table = "silver.orders"
val rows = 4200L
val ratio = 0.127
val dt = "2026-10-04"
```


`f` type-checks the format specifier at compile time. `s` calls `toString`, so an `Option` prints as `Some(...)`; use `.getOrElse("")` before interpolating it into a message.


Worked example of all three interpolators:

```scala
val table = "silver.orders"
val rows = 4200L
val ratio = 0.127
val dt = "2026-10-04"

val line = s"loaded $rows rows into $table on $dt"
val pct  = f"ratio = ${ratio}%.1f%%"
val path = raw"s3://acme-raw/$table/dt=$dt"
val calc = s"double rows = ${rows * 2}"
```

Expected output:
```text
loaded 4200 rows into silver.orders on 2026-10-04
ratio = 12.7%
s3://acme-raw/silver.orders/dt=2026-10-04
double rows = 8400
```



Common mistake: interpolating an `Option` or a `Map` directly into a log line. `s` calls `toString`, so `None` prints as `None` and a `Map` prints braces that confuse log parsers.

```scala
val region: Option[String] = None
println(s"region=$region")                 // region=None
println(s"region=${region.getOrElse("")}") // region=
```

Inside an `s` string a literal dollar sign is written `$$`, and only `$name` or `${expr}` interpolate. A bare `$` in a regex or a SQL string needs escaping.


## How does control flow work in Scala with if/else and match?

`if`/`else` is an expression that returns a value, so it can initialize a `val`. `match` is the general branch construct and replaces long `if`/`else if` chains and `switch`, including guards such as `case Order(_, _, amount, "PAID") if amount > 100`.

```scala
val amount = BigDecimal("1500.00")
val tier = if (amount > 1000) "high" else "standard"
```


Because `OrderEvent` is `sealed`, the compiler warns if a case is missing. That exhaustiveness check is the main reason to model events as sealed hierarchies instead of strings.


Worked example: `match` is an expression, so it returns a value, and guards refine a case.

```scala
sealed trait Event
final case class Order(id: Long, amount: BigDecimal, status: String) extends Event
case object Heartbeat extends Event

def route(event: Event): String = event match {
  case Order(_, amount, "PAID") if amount > 1000 => "priority"
  case Order(_, _, "PAID")                       => "normal"
  case Order(_, _, status)                       => s"review:$status"
  case Heartbeat                                 => "skip"
}
```

Expected: `route(Order(1, BigDecimal(2000), "PAID"))` is `"priority"`, and `route(Heartbeat)` is `"skip"`.



Common mistake: using `match` like a `switch` with a catch-all `case _` first, which silently shadows the real cases. Order matters: specific cases first, catch-all last.

```scala
val status = "PAID"
val label = status match {
  case _      => "unknown"   // matches everything above
  case "PAID" => "paid"      // unreachable
}
```

The compiler warns about the unreachable case. Another common slip is matching a `String` where a sealed type would give exhaustiveness checking, so a new status falls through to `case _` instead of failing the build.


## How do you define functions and methods in Scala?

`def` defines a method and can also define a local function inside another `def`. Scala supports default arguments, named arguments, and varargs written `Int*`.

```scala
def normalize(name: String, trim: Boolean = true, lower: Boolean = true): String = {
  var out = name
  if (trim) out = out.trim
  if (lower) out = out.toLowerCase
  out
}
```


A method returning `Unit` is written `def commit(): Unit = ...`. The older procedure syntax (`def commit() { ... }`) was deprecated in Scala 2.13 and removed in Scala 3, so an interviewer may check whether you still write it.


Worked example of default, named, and varargs parameters:

```scala
def buildPath(bucket: String, dt: String, parts: String*): String =
  (Seq(bucket, dt) ++ parts).mkString("/")

def load(path: String, mode: String = "append", retries: Int = 3): String =
  s"$mode $path retries=$retries"

buildPath("s3://acme", "2026-10-04", "region=us", "hour=07")
load("/data/orders", retries = 5)
```

Expected output:
```text
s3://acme/2026-10-04/region=us/hour=07
append /data/orders retries=5
```



Common mistake: mixing default arguments with overloaded methods. A default argument generates a synthetic method, and overload resolution with defaults gets confusing and sometimes ambiguous.

```scala
def total(amount: BigDecimal, tax: BigDecimal = BigDecimal(0)): BigDecimal = amount + tax
// def total(amount: BigDecimal): BigDecimal = amount   // ambiguous overload
```

Name arguments at the call site when a method has several defaults, and give public methods explicit return types so the inferred type does not surprise callers.


## What are higher-order functions and function values?

Functions are values. `A => B` is a type, and a method that takes or returns a function is higher-order. This is the basis of `map`, `filter`, `fold`, and Spark's typed transformations.

```scala
val double: Int => Int = _ * 2
val amounts = List(10, 20, 30)
```


In Spark, `_.amount` compiles to bytecode instead of a pickled Python lambda, which removes a serialization boundary between the driver and the executors.


Worked example: functions are values passed to combinators.

```scala
val double: Int => Int = _ * 2
val isPositive: Int => Boolean = _ > 0

val amounts = List(10, -5, 20, 0)
val doubled = amounts.filter(isPositive).map(double)
val total = amounts.filter(isPositive).foldLeft(0)(_ + _)

def applyTwice(f: Int => Int, x: Int): Int = f(f(x))
```

Expected: `doubled` is `List(20, 40)`, `total` is `30`, and `applyTwice(double, 3)` is `12`.



Common mistake: writing a lambda with a side effect or a captured `var`, then expecting the effect to be visible on the driver.

```scala
var seen = 0
val counted = List(1, 2, 3).map { n => seen += 1; n * 2 }
// seen is 3 here because this ran on the driver,
// but the same closure shipped to executors would update only a copy.
```

Keep functions pure and return values. A function that only reads its argument and returns a new value is safe to serialize, retry, and test.


## What is currying and partial application?

Currying splits a multi-argument function into a chain of single-argument functions, written as multiple parameter lists. The left lists become reusable configuration and the right list stays the changing data.

```scala
def add(a: Int)(b: Int): Int = a + b
val addTen = add(10)(_)           // Scala 2.13 placeholder; in Scala 3 write add(10)
addTen(5)                         // 15
```


Currying is not the same as default arguments: defaults fill in values at the call site, while currying returns a reusable function value after the first list is applied.


Worked example: the first parameter list is configuration, the second is data.

```scala
def above(min: BigDecimal)(o: Order): Boolean = o.amount >= min
def byStatus(status: String)(o: Order): Boolean = o.status == status

val bigOrders = orders.filter(above(1000))
val paidOrders = orders.filter(byStatus("PAID"))
```



Common mistake: confusing multiple parameter lists with an implicit parameter list. A list marked `implicit` is filled by the compiler, not by partial application.

```scala
def sortRows[A](xs: Seq[A])(implicit ord: Ordering[A]): Seq[A] = xs.sorted
// sortRows(List(3, 1, 2))       // the compiler supplies Ordering[Int]
// sortRows(List(3, 1, 2))(_)    // wrong: the last list is implicit
```

In Scala 2.13 a trailing underscore forces partial application (`add(10)(_)`), while Scala 3 accepts `add(10)` directly. That difference shows up in code review.


## How do case classes and pattern matching work?

A `case class` gets compiler-generated `equals`, `hashCode`, `toString`, `copy`, and companion `apply`/`unapply`, which makes it the standard immutable record type for domain models and Spark encoders.

```scala
final case class Order(orderId: Long, customerId: Long, amount: BigDecimal, status: String)
```


`copy` keeps record updates explicit, which matches Spark's immutable Dataset model. Field order and types must match the source: reading CSV straight into a case class with a `BigDecimal` field fails if the underlying column is a string, so read as a `DataFrame`, cast, then call `.as[Order]`.


Worked example: construct, copy, decompose.

```scala
final case class Order(orderId: Long, customerId: Long, amount: BigDecimal, status: String)

val o = Order(1L, 10L, BigDecimal("19.99"), "pending")
val paid = o.copy(status = "PAID")
val Order(id, _, amount, _) = paid          // pattern binding

def describe(order: Order): String = order match {
  case Order(_, _, a, "PAID") if a > 100 => "large paid order"
  case Order(_, _, _, s)                 => s"status=$s"
}
```

Expected: `paid` equals `o` except `status` is `"PAID"`, `id` is `1L`, `amount` is `19.99`, and `describe` returns `"large paid order"` only when the amount exceeds 100.



Common mistake: expecting `equals` to compare arrays inside a case class. `Array` uses reference equality, so two records with equal contents compare unequal.

```scala
final case class Batch(id: Long, tags: Array[String])
Batch(1L, Array("a")) == Batch(1L, Array("a"))   // false
```

Use `Vector` or `List` for collections inside case classes. Also keep records under the field limit: Scala 2.12 and earlier stop generating `unapply` past 22 fields, so wide records need nesting or a plain class.


## What are sealed traits and extractors?

A `sealed trait` restricts implementations to the same file, so the compiler can prove that a `match` covers every case. That is exactly what you want for event types and job states.

```scala
sealed trait OrderEvent
final case class Created(orderId: Long, customerId: Long) extends OrderEvent
final case class Updated(orderId: Long, amount: BigDecimal) extends OrderEvent
final case class Deleted(orderId: Long) extends OrderEvent
case object RolledBack extends OrderEvent
```


Real-world example: an ingestion job matches `OrderEvent` to route records to insert, merge, or delete paths, and adding a new event type makes the compiler report a non-exhaustive match at every routing site (an error under fatal warnings).


Worked example: an exhaustive match over a sealed hierarchy.

```scala
sealed trait OrderEvent
final case class Created(orderId: Long, customerId: Long) extends OrderEvent
final case class Updated(orderId: Long, amount: BigDecimal) extends OrderEvent
final case class Deleted(orderId: Long) extends OrderEvent
case object RolledBack extends OrderEvent

def target(event: OrderEvent): String = event match {
  case _: Created => "insert"
  case _: Updated => "merge"
  case _: Deleted => "delete"
  case RolledBack => "ignore"
}
```

Expected: `target(Created(1L, 2L))` is `"insert"`, and adding a `case class Refunded` without a branch produces a non-exhaustive match warning.



Common mistake: sealing a trait in one file and extending it in another. Sealing only restricts direct subclasses to the same source file, so a subtype elsewhere breaks the exhaustiveness guarantee and the compiler cannot warn you.

Extractors are the other half: `unapply` decides what a pattern can see. An extractor that returns `Boolean` is for guards, not decomposition.

```scala
object IsPaid {
  def unapply(o: Order): Boolean = o.status == "PAID"
}
order match { case IsPaid() => "paid"; case _ => "other" }
```


## How do traits, mixins, and abstract classes work?

Traits define interfaces plus optional concrete behavior and are mixed in with `with`. A class can extend one class but mix in many traits, which is how data code composes concerns like logging, metrics, and clocks.

```scala
trait Clock { def now(): Long }
trait Logging { def log(message: String): Unit = println(s"[${getClass.getSimpleName}] $message") }
abstract class BaseJob(val name: String) { def run(): Unit }
```


An abstract class is the better choice when you need constructor parameters or Java interoperability; traits are better for stackable behavior. Traits are linearized left to right, so the rightmost mixin wins for conflicting concrete methods, which is why `abstract override` is needed to wrap `super.run()`.


Worked example: stackable traits that wrap behavior.

```scala
trait Job { def run(): String = "run" }
trait Timed extends Job {
  abstract override def run(): String = {
    val start = System.nanoTime()
    val out = super.run()
    s"$out (${(System.nanoTime() - start) / 1000000} ms)"
  }
}
trait Logged extends Job {
  abstract override def run(): String = s"log:${super.run()}"
}

class OrdersJob extends Job with Timed with Logged
```

Expected: `new OrdersJob().run()` starts with `"log:"`, because `Logged` is rightmost in the linearization and wraps `super.run()`.



Common mistake: putting mutable state in a trait. Every mixing class gets its own copy, and initialization order follows the constructor order, which surprises people.

```scala
trait Counter {
  var count = 0            // one slot per instance, initialized late
  def bump(): Unit = count += 1
}
```

Prefer a `val` computed from constructor arguments, or a method that takes state as a parameter. A trait holding a `SparkSession` is a common source of serialization errors, because the session is not serializable and gets captured in closures.


## What are objects and companion objects with apply and unapply?

An `object` is a singleton. An object with the same name as a class or case class in the same file is that class's companion, and it can see the class's private members. Companions are the idiomatic home for factories, constants, and `unapply`.

```scala
final case class Partition(dt: String, region: String)
```


`apply` is syntax sugar for calling the object (`obj(args)` means `obj.apply(args)`), and `unapply` is what pattern matching calls to take a value apart. Defining both on a companion gives a plain class case-class-like ergonomics.


Worked example: a factory plus a custom extractor on the companion.

```scala
final class Partition private (val dt: String, val region: String)

object Partition {
  def apply(dt: String, region: String = "us"): Partition =
    new Partition(dt.trim, region.toLowerCase)

  def unapply(p: Partition): Option[(String, String)] = Some((p.dt, p.region))
}

val p = Partition("2026-10-04")            // calls Partition.apply
val Partition(dt, region) = p              // calls Partition.unapply
val label = p match {
  case Partition(d, "us") => s"$d/us"
  case Partition(d, r)    => s"$d/$r"
}
```

Expected: `dt` is `"2026-10-04"`, `region` is `"us"`, and `label` is `"2026-10-04/us"`.



Common mistake: assuming an `object` is created eagerly in a fixed order. An object body runs on first access, and cyclic initialization between two objects can leave one with a `null` field.

```scala
object A { val x = B.y + 1 }
object B { val y = A.x + 1 }   // one of these sees the other mid-initialization
```

Keep companion objects as pure factories, constants, and extractors with no cross-object initialization. Note also that a singleton object captured in a Spark closure is serialized per task.


## How do Scala collections and core combinators work?

`List` is a linked list with cheap head/tail, `Vector` is the indexed immutable sequence, `Seq` is the general sequence abstraction, and `Map`/`Set` are the associative and unique collections. In Scala 2.13 the default `Seq` is `scala.collection.immutable.Seq`.

```scala
val statuses: List[String] = List("PAID", "PENDING", "PAID")
val regions: Set[String] = Set("us", "eu", "us")
```


`fold`/`foldLeft` carries an initial accumulator and can change the result type; `reduce` seeds from the first element and fails on empty input, which is why `reduceOption` exists. Use `.view.mapValues(...)` to transform a `Map` lazily instead of rebuilding it eagerly.


Worked example: the core combinators and what each returns.

```scala
val amounts = List(10, 20, 30)
val doubled = amounts.map(_ * 2)                 // List(20, 40, 60)
val kept = amounts.filter(_ > 15)                // List(20, 30)
val total = amounts.reduce(_ + _)                // 60
val summed = amounts.foldLeft(0)(_ + _)          // 60, result type Int
val running = amounts.scanLeft(0)(_ + _)         // List(0, 10, 30, 60)
val byParity = amounts.groupBy(_ % 2 == 0)       // Map(false -> List(10, 30), true -> List(20))
```



Common mistake: using `reduce` on a collection that may be empty, or reaching for a mutable collection when a `fold` is clearer.

```scala
List.empty[Int].reduce(_ + _)          // throws: empty.reduceLeft
List.empty[Int].reduceOption(_ + _)    // None, safe
```

Know the API split too. `scala.collection.immutable.List` has no `+=`; adding means creating a new list. The mutable variants live in `scala.collection.mutable` and must be imported explicitly. In 2.13, `mapValues` on a `Map` returns a lazy view, so use `.view.mapValues(...).toMap` to make the intent explicit.


## How do for-comprehensions desugar in Scala?

A `for` expression is syntax sugar: generators become `flatMap`, the final `yield` becomes `map`, filters become `withFilter`, and a `for` without `yield` becomes `foreach`.

```scala
val pairs = for {
  a <- List(1, 2, 3)
  b <- List(10, 20) if a < 3
} yield (a, b)
```


Because the desugaring only needs those methods, the same syntax sequences `Option`, `Either`, and `Future`:

```scala
def lookupCustomer(id: Long): Option[String] = if (id == 10L) Some("alice") else None
```


Repository-style lookups return `Option`, so a chain of enrichment steps reads top to bottom and short-circuits on the first missing value instead of nesting `if` blocks or throwing.


Worked example: the desugaring written out.

```scala
val pairs = for {
  a <- List(1, 2, 3)
  b <- List(10, 20) if a < 3
} yield (a, b)

// equivalent to:
val same = List(1, 2, 3).flatMap { a =>
  List(10, 20).withFilter(_ => a < 3).map(b => (a, b))
}
```

Expected: `pairs` is `List((1,10), (1,20), (2,10), (2,20))`, because the filter applies once per `a`.



Common mistake: reading a nested `for` as a plain loop. Because each generator becomes `flatMap`, mixing a collection generator with a `Future` generator in one comprehension does not compile.

```scala
// does not compile: List has no flatMap back to Future
// for { x <- List(1, 2); y <- Future(x + 1) } yield x + y
```

Keep one effect per comprehension. Use `for` over `Option` for enrichment chains, and `Future.sequence` to combine a collection of futures instead of mixing them.


## How do you use Option, Either, and Try instead of null?

Scala encodes failure in the type system. `Option[A]` means "a value or nothing", `Either[L, R]` carries a typed error on the left, and `Try[A]` captures a thrown exception.

```scala
import scala.util.{Failure, Success, Try}
```


Why this matters at data boundaries:


Worked example of the three types and how they compose.

```scala
import scala.util.{Failure, Success, Try}

val maybe: Option[String] = Option(System.getenv("REGION"))    // null becomes None
val region = maybe.getOrElse("us")

def parseAmount(raw: String): Either[String, BigDecimal] =
  Try(BigDecimal(raw.trim)).toEither.left.map(_ => s"bad amount: $raw")

val doubled: Either[String, BigDecimal] = parseAmount("19.99").map(_ * 2)
val safe: Try[Int] = Try("42".toInt)
```

Expected: `region` is `"us"` when the variable is unset, `parseAmount("19.99").map(_ * 2)` is `Right(39.98)`, and `parseAmount("x")` is a `Left` with a readable message.



Common mistake: calling `.get` on an `Option` or a `Try` to "keep it simple", which throws at runtime and defeats the type.

```scala
val maybe: Option[String] = None
// maybe.get          // throws NoSuchElementException
maybe.getOrElse("us") // safe default
maybe match {
  case Some(v) => v
  case None    => "us"
}
```

Prefer `getOrElse`, `map`, `flatMap`, `fold`, or a `match`. Convert at boundaries: `Option(x)` for null-returning Java calls, `Try(expr).toEither` for parsing, and `Either.left.map` to attach context instead of an opaque exception.


+ Java APIs, JDBC, and environment variables return `null`; wrap them immediately with `Option(...)`
+ Spark `Dataset[T]` rows can still contain `null` in a case class field, so validate with a filter before writing a governed table

## What are implicits, givens, and extension methods?

Implicits are Scala 2's mechanism for automatically passing values and applying conversions. Scala 3 renamed the concept to `given`/`using` and added `extension` methods. Both add behavior to types you do not own, most commonly Spark's `Dataset`.

Scala 2.13 form, which is what Spark 3.x and Spark 4.x modules compile against:

```scala
import org.apache.spark.sql.Dataset
```


Scala 3 form of the same idea, plus an implicit type class instance:

```scala
// Scala 3 only: Spark core artifacts are published for Scala 2.12/2.13, not for Scala 3
extension (ds: Dataset[Order])
  def logCount(): Dataset[Order] = { println(s"rows=${ds.count()}"); ds }
```


Be honest about versions: `implicit` is the safe answer for Spark work today, while `given`/`extension` need a Scala 3 toolchain. Keep implicits narrow and local, because broad implicit conversions slow compilation and make type errors hard to read.


Outline of a type class in Scala 2.13 style: a trait with the operations, instances for concrete types, and a method that asks for the instance.

```scala
trait Encoder2[A] { def encode(a: A): String }
object Encoder2 {
  implicit val intEncoder: Encoder2[Int] = (a: Int) => a.toString
  implicit val orderEncoder: Encoder2[Order] = (o: Order) => s"${o.orderId},${o.amount}"

  implicit class Ops[A](private val a: A) extends AnyVal {
    def encode(implicit e: Encoder2[A]): String = e.encode(a)
  }
}

import Encoder2._
Order(1L, BigDecimal("9.99"), "PAID").encode   // instance supplied by the compiler
```

Expected: the `.encode` call resolves `orderEncoder` at compile time and returns `"1,9.99"`.



Common mistake: adding an implicit conversion between two broad types, then spending hours on ambiguous implicit errors.

```scala
implicit def strToInt(s: String): Int = s.toInt   // dangerous: any String can become Int
```

Keep implicits narrow: a type class instance for one concrete type, a context bound, or an `implicit class` for syntax. Do not implicitly convert between domain types. If two instances of the same type are in scope, the compiler reports an ambiguity, so import instance objects explicitly instead of wildcard-importing several of them.


## How do generics and type bounds work in Scala?

Generic methods and classes use type parameters in square brackets, like Java generics but with declaration-site variance and richer bounds.

```scala
trait Source[+A] { def read(): A }          // covariant: produces A
trait Sink[-A]   { def write(a: A): Unit }  // contravariant: consumes A
```


Bounds that show up most in data code:


Worked example of a context bound and an upper bound.

```scala
def largest[A: Ordering](xs: Seq[A]): Option[A] =
  if (xs.isEmpty) None else Some(xs.max)

trait HasAmount { def amount: BigDecimal }
def sumAmounts[A <: HasAmount](xs: Seq[A]): BigDecimal = xs.map(_.amount).sum

largest(List(3, 9, 4))   // Some(9)
largest(List("b", "a"))  // Some("b")
```

Expected: `largest(List(3, 9, 4))` is `Some(9)`, and the compiler supplies `Ordering[Int]` and `Ordering[String]`.



Common mistake: expecting the JVM to keep generic type arguments at runtime. Generics are erased, so `List[Int]` and `List[String]` are both `List` at runtime.

```scala
val xs: List[Int] = List(1, 2, 3)
xs match {
  case _: List[String] => "strings"     // compiles, misleading
  case _               => "other"
}
```

When generic code needs runtime class information, ask for a `ClassTag` or an `Encoder`. For typed Spark datasets the `Encoder[T]` is the runtime evidence that the type is known, which is why `Dataset[T]` operations require one.


+ `<: A` upper bound requires a subtype, for example `Serializable` before shipping data across the network
+ `: Ordering` context bound supplies an implicit `Ordering[A]` so `sortBy` and `max` work
+ `: ClassTag` context bound supplies runtime class information, needed for arrays or encoders in generic code

## What are lazy val and by-name parameters?

A `lazy val` is initialized on first access and memoized. A by-name parameter (`=> T`) defers evaluation until the method body uses it, and re-evaluates on each use unless you cache it.

```scala
lazy val spark = SparkSession.builder().appName("orders").getOrCreate()
```


By-name parameters power assertions, retries, and cheap logging. In Spark jobs, `lazy val` for the `SparkSession` and for shared schemas avoids creating sessions during unit tests that never touch Spark. `lazy val` is thread-safe by default, so concurrent first access blocks until initialization completes.


Worked example: by-name defers and re-evaluates, `lazy val` defers once.

```scala
def timed[T](label: String)(body: => T): T = {
  val start = System.nanoTime()
  val out = body
  println(s"$label ${(System.nanoTime() - start) / 1000000} ms")
  out
}

lazy val schema = { println("building schema"); "order_id BIGINT" }

timed("query")(runQuery())     // body runs once, inside the timer
schema                          // prints "building schema" on first use
schema                          // second use does not print
```

Expected: the timer wraps exactly one execution of `runQuery()`, and the schema is built once.



Common mistake: assuming a by-name parameter evaluates once. Every use in the body runs the expression again.

```scala
def twice(x: => Int): Int = x + x    // evaluates x twice
twice { println("expensive"); 21 }   // prints twice
```

Cache it in a `val` inside the method if it must run once. The mirror-image mistake is cyclic `lazy val` initialization, which deadlocks or throws at first access because two lazy values wait on each other.


## How do Futures and basic concurrency work in Scala?

`Future[T]` is a value that will be available later. Computation starts when the `Future` is created, runs on an `ExecutionContext`, and is composed with `map`, `flatMap`, and `Future.sequence`.

```scala
import scala.concurrent.{Await, ExecutionContext, Future}
import scala.concurrent.duration._
```


Rules that matter in data pipelines:


Worked example: compose independent futures and await once at the edge.

```scala
import scala.concurrent.{Await, Future}
import scala.concurrent.ExecutionContext.Implicits.global
import scala.concurrent.duration._

def writePartition(part: String): Future[Long] =
  Future { /* submit one Spark write */ 1L }   // rows written

val partitions = Seq("us", "eu", "apac")
val writes: Future[Seq[Long]] = Future.sequence(partitions.map(writePartition))
val total = Await.result(writes, 10.minutes).sum
```

Expected: `writes` completes only when all three writes complete, and one failure fails the whole `Future`. A single `Await` at the job boundary keeps blocking out of the composition.



Common mistake: creating a `Future` inside a loop to parallelize hundreds of Spark submissions. Each `Future` starts immediately, so an unbounded fan-out can flood the scheduler.

```scala
// risky: 5000 concurrent submissions
val all = partitions.map(writePartition)   // each starts now
```

Batch or throttle the work. Also avoid the global execution context for blocking JDBC or file IO; use a dedicated fixed pool sized to the connection limit, and never `Await` inside a `map` on that same pool, because the pool can starve.


+ Do not use the global execution context for blocking work such as JDBC or file IO; size a dedicated pool, and bound fan-out because hundreds of concurrent Spark job submissions from one driver is worse than batching

## How does Java interoperability work in Scala?

Scala runs on the JVM, calls Java directly, and extends Java classes. The friction is mostly collection conversions and `null`, which is why `scala.jdk.CollectionConverters` exists.

```scala
import java.time.Instant
import java.util.{List => JList, Properties}
import scala.jdk.CollectionConverters._
```


Interop points interviewers check:


Worked example: pass a Java `Properties` object to Kafka and read a Java list back into Scala.

```scala
import java.util.{Collections, Properties}
import scala.jdk.CollectionConverters._

val props = new Properties()
props.put("bootstrap.servers", "kafka:9092")
props.put("group.id", "orders-loader")

val topics: java.util.List[String] = Collections.singletonList("orders")
val topicNames: Seq[String] = topics.asScala.toSeq

val fromJava: String = System.getProperty("user.home")   // may be null
val safe: Option[String] = Option(fromJava)
```

Expected: `topicNames` is `Seq("orders")`, and a null system property becomes `None` instead of throwing later.



Common mistake: forgetting that Java boxed types are not Scala primitives. A Java method returning `Integer` can return `null`, while `Int` cannot.

```scala
val boxed: java.lang.Integer = null
// val n: Int = boxed        // unboxing null throws NullPointerException
val n: Option[Int] = Option(boxed).map(_.intValue)
```

Convert collections explicitly with `asScala` and `asJava` instead of relying on the deprecated `JavaConverters`. Scala `List` is not a `java.util.List`, and passing the wrong one compiles only when an implicit conversion is in scope.


+ `scala.jdk.CollectionConverters._` is current in 2.13 and 3; the older `JavaConverters` is deprecated
+ Java methods that can return `null` should be wrapped with `Option(...)` at the call site
+ Scala annotations such as `@BeanProperty` and `@throws` expose Java-visible members when Java must consume the class
+ Java builders and setters stay as they are, which is why Kafka configuration uses `props.put(...)` rather than Scala collections

## How do you structure an sbt project with dependencies and tests?

sbt is the standard Scala build tool. A job project has a small, predictable layout, and the build file is itself Scala code.

```text
my-job/
  build.sbt
  src/main/scala/com/acme/OrdersJob.scala
  src/test/scala/com/acme/OrdersJobSpec.scala
```

```scala
// build.sbt
ThisBuild / scalaVersion := "2.13.14"
```


`%%` appends the Scala binary version (resolving `spark-sql_2.13`), and Spark dependencies are `Provided` because the cluster already ships them. `Test / fork := true` gives tests their own JVM so Spark and sbt classloaders do not conflict. Tests use MUnit below; ScalaTest works the same way with `AnyFunSuite` and `assert`.

```scala
// src/test/scala/com/acme/OrdersJobSpec.scala
import munit.FunSuite
```


Build and test with `sbt compile`, `sbt test`, `sbt "runMain com.acme.OrdersJob --dt 2026-10-04"`, or `sbt publishLocal` to share the transforms library with other jobs.


Worked example of a small, complete build file:

```scala
// build.sbt
ThisBuild / scalaVersion := "2.13.14"

lazy val root = (project in file("."))
  .settings(
    name := "orders-job",
    libraryDependencies ++= Seq(
      "org.apache.spark" %% "spark-sql" % "3.5.1" % Provided,
      "org.scalameta"    %% "munit"     % "1.0.0" % Test
    ),
    Test / fork := true,
    scalacOptions ++= Seq("-deprecation", "-feature", "-Xfatal-warnings")
  )
```

Expected: `sbt test` compiles `src/main/scala` and `src/test/scala`, runs the MUnit suites, and `sbt assembly` produces a jar without Spark inside it.



Common mistake: publishing a job assembled with Spark inside it, or mixing Scala binary versions. `Provided` keeps Spark out of the assembly, and `%%` resolves the artifact for the project's Scala version, so `spark-sql` becomes `spark-sql_2.13` or `spark-sql_2.12`.

```scala
// wrong version for the cluster: builds fine, then fails at runtime with NoSuchMethodError
"org.apache.spark" %% "spark-sql" % "3.5.1" % Provided
```

A `NoSuchMethodError` at runtime usually means a dependency was built for a different Scala binary version than the cluster runtime. Check `sbt dependencyTree` before blaming the cluster.


## What Scala idioms matter for Spark?

Spark's Scala API exposes typed `Dataset[T]`, `Encoders`, strongly typed aggregations, and direct access to internals. The main design decision is typed versus untyped.

```scala
import org.apache.spark.sql.{Dataset, SparkSession}
import org.apache.spark.sql.functions.{broadcast, col, udf}
```


Typed versus untyped:

+ `Dataset[T]` with case classes gives compile-time field checks and `Encoder[T]` derivation via `import spark.implicits._`
+ `DataFrame` (`Dataset[Row]`) with `Column` handles runtime-only schemas, is usually better optimized because Catalyst sees the whole expression tree, and suits wide or dynamic data

When Scala beats PySpark: CPU-heavy row-wise logic, logic shared as a JVM library with Kafka or Flink, and teams that want schema errors at build time. PySpark still wins on notebook iteration, the Python ML ecosystem, hiring friction, and built-in transformations where the JVM difference is small. Use a UDF only when no built-in function exists, and broadcast the small side of a join.


Worked example: the same transform typed and untyped.

```scala
import org.apache.spark.sql.{Dataset, SparkSession}
import org.apache.spark.sql.functions.{col, sum}

final case class Order(orderId: Long, region: String, amount: BigDecimal)

def positive(orders: Dataset[Order]): Dataset[Order] =
  orders.filter(_.amount > 0)                 // typed lambda, checked at build time

val df = spark.read.parquet("s3://acme/silver/orders")
df.filter(col("amount") > 0)
  .groupBy("region")
  .agg(sum("amount").as("revenue"))           // untyped, but Catalyst sees everything
```

Expected: both plans produce the same physical aggregation. The typed version fails to compile if `Order` loses a field; the untyped version fails only at runtime.



Common mistake: wrapping simple logic in a UDF. A UDF is a black box to Catalyst, so it blocks predicate pushdown, code generation, and column pruning, and in Scala it also has to be serializable.

```scala
import org.apache.spark.sql.functions.udf
val plusOne = udf((x: Int) => x + 1)             // opaque to the optimizer
orders.withColumn("n", plusOne(col("amount")))   // prefer col("amount") + 1
```

Reach for a UDF only when no built-in expression does the job, and prefer the typed `Dataset` API when you want the compiler to check field names.


## What are common Scala pitfalls in data engineering?

+ `null` leaks from Java and from Spark rows. Wrap with `Option(...)` at the boundary, and remember that a `null` inside a `Dataset[T]` field does not throw until you call a method on it
+ `==` versus `equals`: `==` calls `equals` and is null-safe, so `a == b` is usually right. It is structural for case classes but reference-based for `Array`, where `sameElements` is what you want; `eq` is reference identity
+ Mutable collections captured in closures are copied to executors, so mutations are lost. Keep transformations pure and return values
+ Implicit conversions hide real errors and slow compilation; Scala 3 requires `import scala.language.implicitConversions` to use them, which is a hint to avoid them
+ Read stack traces bottom-up. Start from the deepest `Caused by` and the first frame in your own package; Spark wraps failures in `SparkException`, so the useful line is usually several causes down
+ Version mismatch is a real production failure: Spark 3.x artifacts exist for Scala 2.12 and 2.13, while Spark 4.x dropped 2.12 and publishes 2.13 only; adding a Scala 3 or mismatched 2.13 library to a 2.12 cluster gives `NoSuchMethodError` at runtime
+ Scala 2 versus Scala 3: `implicit`/`implicit class` versus `given`/`using`/`extension`; `_` versus `?` for wildcard types; procedure syntax and symbol literals removed; in 2.13 `Seq` defaults to `scala.collection.immutable.Seq`

A defensive pattern for the `null` pitfall:

```scala
def validate(o: RawOrder): Either[String, RawOrder] = {
  if (o.status == null || o.status.trim.isEmpty) Left(s"missing status for order ${o.orderId}")
  else if (o.amount == null || o.amount <= 0)    Left(s"bad amount for order ${o.orderId}")
  else Right(o.copy(status = o.status.trim.toUpperCase))
}
```

Route `Left` values to a quarantine table and continue, rather than throwing and failing the whole partition.


Worked example: a mutable closure across a parallel boundary.

```scala
var dropped = 0
orders.filter { o =>
  if (o.amount <= 0) { dropped += 1; false } else true
}
println(s"driver sees dropped=$dropped")   // stays 0 on the driver
```

Expected: on a distributed run `dropped` stays `0` on the driver, because each executor mutates its own serialized copy. Return a tagged result instead and count with an aggregation.



Common mistake: comparing arrays, or mistaking reference identity for structural equality.

```scala
val a = Array(1, 2, 3)
val b = Array(1, 2, 3)
a == b                    // false: reference equality for Array
a.sameElements(b)         // true
a eq b                    // false: reference identity
List(1, 2) == List(1, 2)  // true: structural equality
```

`==` calls `equals` and is null-safe, which is correct for case classes, `List`, and `Option`. Use `sameElements` for arrays, and `eq` only when you truly mean "the same object".


## How do you write Scala that performs well in Spark?


Performance work in Spark is mostly about not doing extra work: not copying data you do not change, not serializing objects you do not need, and not handing the optimizer a black box. The three sections below cover the Scala-side decisions that show up in job tuning.

```text
driver: build plan -> serialize closures -> executors: run tasks -> shuffle -> collect
         val, no capture     encoders          built-ins, no per-row objects
```


## How do immutability and copying costs affect a Spark job?


Immutable collections are cheap to share because nobody can change them in place. The cost is that an update builds a new structure. For `List` the new head shares the tail, so prepending is O(1), while appending is O(n). `Vector` uses a tree and gives effectively constant-time update with structural sharing.

```scala
val xs = List(1, 2, 3)
val ys = 0 :: xs            // shares the tail, O(1)
val zs = xs :+ 4            // copies the spine, O(n)
val vs = Vector(1, 2, 3)
val ws = vs.updated(1, 99)  // shares most nodes, effectively O(log n)
```

Expected: `ys` is `List(0, 1, 2, 3)`, `zs` is `List(1, 2, 3, 4)`, and `ws` is `Vector(1, 99, 3)`.



Common mistake: rebuilding a large collection on every step, especially appending to a `List` inside a loop.

```scala
// O(n^2): each :+ copies the whole list
var out = List.empty[Int]
(1 to 100000).foreach(i => out = out :+ i)

// O(n): build once with a Vector
val fast = (1 to 100000).toVector
```

When a task must accumulate a large result, a local `mutable.ArrayBuffer` or `Vector.newBuilder` inside the task is the right call. Keep the mutation inside one function, return an immutable value or an iterator, and never let the mutable object escape into a closure or an accumulator.



The rule of thumb: immutable at the boundaries, mutable inside a hot local loop. A builder that stays inside one task cannot create the lost-update bug that a captured `var` creates. Convert to an immutable collection before returning.


## What has to be serializable in a Spark closure?


A Spark transformation is a closure. The driver serializes it and ships it with each task. Everything the closure touches has to be serializable: the case classes in a `Dataset[T]` via their `Encoder`, the function values, and any field the enclosing object or lambda captures.

```scala
final case class Order(orderId: Long, amount: BigDecimal)   // serializable by default

// captures nothing that is not serializable
val highValue = orders.filter(_.amount > 1000)

// captures `spark`, which is not serializable
class BadJob(spark: SparkSession) {
  def run(df: Dataset[Order]) =
    df.filter(o => spark.sparkContext.isLocal && o.amount > 0)
}
```

Expected: the first filter ships cleanly. The second fails at task submission with `org.apache.spark.SparkException: Task not serializable`, whose cause is a `NotSerializableException` naming the class.



Common mistake: capturing the enclosing object, often a job class that holds a `SparkSession`, because a lambda refers to one field. In Scala a lambda inside a class method captures the whole `this`.

```scala
class Job(val spark: SparkSession) {
  val minAmount = BigDecimal(1000)                 // fine on its own
  def run(df: Dataset[Order]) =
    df.filter(o => o.amount > minAmount)           // captures this, which holds spark
}
```

Fix it by copying the needed value into a local `val` before the closure, or by making the helper a function that takes the value as a parameter. Case classes, `String`, numbers, and `Vector` serialize; `SparkSession`, `SparkContext`, `DataFrame`, and open JDBC connections do not.



How to read the failure: `Task not serializable` is the symptom, and the first `Caused by: java.io.NotSerializableException` names the class that could not be written. Search the driver stack trace for your own package and the line that creates the closure.

```text
org.apache.spark.SparkException: Task not serializable
Caused by: java.io.NotSerializableException: com.acme.Job
  at java.io.ObjectOutputStream.writeObject0(...)
```


## How do collections and type choices affect Spark job performance?


Three choices dominate. First, represent missing data as `Option` in domain code but avoid `Option` inside hot per-row objects in a `Dataset`, because every `Some` is an allocation; a nullable column is cheaper and the schema documents it. Second, use built-in `functions` expressions instead of UDFs so Catalyst can optimize. Third, avoid creating an object per row in a tight loop.

```scala
import org.apache.spark.sql.functions._

// built-in: stays inside Catalyst, gets code-generated
orders.withColumn("amount_cents", (col("amount") * 100).cast("long"))
      .filter(col("amount") > 0)

// UDF: opaque, serialized per task, no pushdown
val cents = udf((a: java.math.BigDecimal) => a.multiply(java.math.BigDecimal.valueOf(100)))
orders.withColumn("amount_cents", cents(col("amount")))
```

Expected: both compute the same value, but the built-in version can push the filter into the scan and skip rows before they reach the projection.



Common mistake: assuming `Dataset[T]` is always faster than `DataFrame`. The typed API is friendlier but adds encoder work; `Dataset[Row]` with `Column` expressions is often faster because Catalyst sees the whole expression tree.

```scala
// typed: lambda is opaque to Catalyst until it is converted
orders.filter(_.amount > 0)
// untyped: fully visible to the optimizer
orders.filter(col("amount") > 0)
```

Collect to the driver only when the result is small. A `collect()` that pulls millions of rows, or a `groupByKey` that builds an unbounded per-key collection, is usually the real performance bug, not the language.


## Real world use cases for Scala in data engineering


The five sketches below cover the shapes that come up most in interviews: a typed Spark pipeline, a Kafka consumer, a Flink job, a shared transforms library, and a bounded-concurrency streaming step. For each one, be ready to say which part is Scala-specific and which part would look the same in Java or Python.


### 1. Typed Spark Dataset pipeline

Case classes plus `Dataset[T]` make the transform contract explicit, and `partitionBy` keeps output rewritable per day.

```scala
import org.apache.spark.sql.{Dataset, SparkSession}
```


Why Scala: a renamed or retyped case class field fails the build instead of the midnight run, and `transform` is unit-testable on a local `SparkSession` with a small `Seq` input.

### 2. Kafka consumer that lands raw events

The official Kafka client is Java, so Scala consumes it directly and pattern matches on the decoded payload for routing.

```scala
import java.time.Duration
import java.util.{Collections, Properties}
import org.apache.kafka.clients.consumer.{ConsumerConfig, KafkaConsumer}
import org.apache.kafka.common.serialization.StringDeserializer
import scala.jdk.CollectionConverters._
```


Why Scala: disabling auto-commit and committing after the batch is durable gives at-least-once delivery with a controlled checkpoint, and the consumer can ship in the same JVM artifact as the transformation library.

### 3. Flink job sketch

Flink exposes a JVM DataStream API with a Scala layer, so a Scala service can own a streaming job end to end.

```scala
import org.apache.flink.api.common.eventtime.WatermarkStrategy
import org.apache.flink.api.common.serialization.SimpleStringSchema
import org.apache.flink.connector.kafka.source.KafkaSource
import org.apache.flink.connector.kafka.source.enumerator.initializer.OffsetsInitializer
import org.apache.flink.streaming.api.scala._
```


Why Scala: the job is a normal JVM deployment with the same build, logging, and tests as the rest of the platform. Version caveat: Flink's Scala DataStream API has been deprecated and removed in recent releases, so prefer the Java API for new projects.

### 4. Shared transformation library for batch and streaming

One library keeps batch and streaming definitions from drifting apart.

```scala
// src/main/scala/com/acme/transforms/OrderTransforms.scala
package com.acme.transforms
```


Published with `sbt publishLocal` or to the company artifact repository, then pulled into the batch job and the streaming job as one versioned dependency, so a revenue-rule change is reviewed and released once.

### 5. Streaming ETL with Futures and bounded batching

A micro-batch streaming ETL step consumes a checkpointed stream, and catch-up or replay work is driven with `Future` batching so the driver stays in control of write parallelism.

```scala
import scala.concurrent.{Await, Future}
import scala.concurrent.ExecutionContext.Implicits.global
import scala.concurrent.duration._
```


Why Scala: bounded batches stop a driver from submitting thousands of concurrent jobs while draining a backlog, completion is tracked per batch so a failed group can be retried, and the row counts feed data-quality checkpoints.

### Quick check

Answer these out loud before you move on.

1. Why should a Spark transformation close over a `val` rather than a `var`?
2. What does `%%` do in `build.sbt`, and why are Spark dependencies marked `Provided`?
3. A teammate adds a library compiled for Scala 3 to a Spark job on a Scala 2.12 cluster and gets `NoSuchMethodError` at runtime. Why?
4. When is `a == b` the wrong comparison for two collections?

**Answers.**

1. Closures are serialized and copied to executors, so a captured `var` is a private copy and executor updates never reach the driver. An immutable `val` removes that failure mode and is safe to share across threads and Futures.
2. `%%` appends the Scala binary version to the artifact name, resolving `spark-sql_2.13`. `Provided` marks Spark as supplied by the cluster at runtime, so the assembly does not package it.
3. Scala 2 and Scala 3 artifacts are binary-incompatible, and Spark 3.x is published only for 2.12 and 2.13. The method signature the library expects does not exist in the cluster's runtime, so the link fails.
4. `Array` compares by reference with `==`, so use `sameElements` there. Case classes get structural `equals`, which is why `==` is correct for them.


## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** Why do data engineers learn Scala instead of staying in PySpark?

**A.** Spark and Flink are Scala and JVM first, so source code, error messages, and internals are easier to follow, and typed `Dataset[T]` code fails at compile time instead of at midnight. One Scala library can be shared by batch jobs, streaming jobs, and microservices, and JVM logic avoids Python UDF serialization for CPU-heavy row work. PySpark still wins for notebook iteration, the Python ML ecosystem, hiring, and built-in transformations.

**Q2.** What is the difference between `val` and `var`, and what does `val` actually guarantee?

**A.** `val` binds a name once; `var` can be reassigned. Default to `val` and use `var` only for tight local accumulation. `val` makes the reference immutable, not the object, so a `val` can still point at a mutable collection.

**Q3.** How does a `for` comprehension desugar, and why does it work on more than collections?

**A.** Generators become `flatMap`, the final `yield` becomes `map`, filters become `withFilter`, and a `for` without `yield` becomes `foreach`. Because the desugaring only needs those methods, the same syntax sequences `Option`, `Either`, and `Future`.

**Q4.** When do you reach for `Option`, `Either`, and `Try`?

**A.** `Option[A]` models a value that may be absent, `Either[L, R]` carries a typed error on the left for expected failures, and `Try[A]` captures a thrown exception. Wrap null-returning Java, JDBC, and environment lookups in `Option(...)` at the boundary, and route `Left` values to a quarantine table instead of throwing and failing the partition.

**Q5.** What does a `sealed trait` buy you in an ingestion job?

**A.** Sealing restricts implementations to the same file, so the compiler can prove a `match` covers every case and warns when a new event type is added without updating a routing site. That turns a forgotten delete or rollback case into a build error rather than a runtime surprise.

**Q6.** What is a `lazy val`, and why does it matter for a `SparkSession`?

**A.** A `lazy val` is initialized on first access and memoized, and initialization is thread-safe. Making the `SparkSession` and shared schemas lazy means unit tests that never touch Spark do not pay for session startup, and concurrent first access still initializes only once.

**Q7.** Why does a mutable collection captured in a Spark or Flink closure behave differently on the driver and on an executor?

**A.** The closure is serialized and shipped, so the executor mutates its own copy and the driver never sees the update. Keep transformations pure by returning new values, and bound `Future` fan-out from the driver because hundreds of concurrent submissions are worse than batching.



Use these questions as a self-test, not a reading list. Answer out loud first, then compare. If an answer is thin, go back to the section named in the question and write the one-line version in your own notes.

The runnable cells at the end of this notebook are Python models of the Scala mechanics, so you can check your reasoning even without a JVM.


## Scala exercises


These exercises model the Scala mechanics in standard-library Python so they run without a JVM. Each one states the task, a hint, and the expected result before the code cell.


### Try it: immutable update versus a mutable builder

A Scala `case class` update with `copy` allocates a new record. A `var` plus a mutable map updates in place. Model both on a list of dicts and count how many records each strategy allocates.

Task: apply `status = "PAID"` to 5,000 records with (a) a pure function that returns new dicts and (b) an in-place update of existing dicts. Count allocations in each case.

Hint: a new dict is one allocation; the in-place path allocates nothing per record.

Expected result: the pure version allocates 5,000 records, the mutable version allocates 0, and both end with the same status.


In [1]:
import copy

records = [{"id": i, "status": "PENDING"} for i in range(5000)]

allocations = 0


def pure_update(rows):
    global allocations
    out = []
    for row in rows:
        new_row = dict(row)          # a copy, like case class .copy
        new_row["status"] = "PAID"
        allocations += 1
        out.append(new_row)
    return out


pure_result = pure_update(records)
assert allocations == 5000
assert records[0]["status"] == "PENDING"          # originals untouched
assert all(r["status"] == "PAID" for r in pure_result)

mutable_allocations = 0


def mutable_update(rows):
    global mutable_allocations
    for row in rows:
        row["status"] = "PAID"       # in place, like a mutable builder
    return rows


mutable_result = mutable_update(records)
assert mutable_allocations == 0
assert mutable_result[0] is records[0]
assert records[0]["status"] == "PAID"

print("pure allocations:", allocations)
print("mutable allocations:", mutable_allocations)
print("pure keeps originals; mutable rewrites them in place")


pure allocations: 5000
mutable allocations: 0
pure keeps originals; mutable rewrites them in place


### Try it: rewrite a nested loop as flatMap/filter

A Scala `for` with two generators desugars to `flatMap` followed by `map`, and a guard becomes a filter. Model a two-list join and count how many pairs each form visits.

Task: generate all pairs from `List(1, 2, 3)` and `List(10, 20)` where `a < 3`.

Hint: `flat_map(xs, f)` is `[y for x in xs for y in f(x)]`.

Expected result: pairs are `(1,10), (1,20), (2,10), (2,20)`, and both forms visit the same 6 candidate pairs.


In [2]:
def flat_map(xs, f):
    out = []
    for x in xs:
        out.extend(f(x))
    return out


a_values = [1, 2, 3]
b_values = [10, 20]

nested = []
nested_visits = 0
for a in a_values:
    for b in b_values:
        nested_visits += 1
        if a < 3:
            nested.append((a, b))

pipeline_visits = 0


def candidates(a):
    global pipeline_visits
    result = []
    for b in b_values:
        pipeline_visits += 1
        result.append((a, b))
    return result


pipeline = [pair for pair in flat_map(a_values, candidates) if pair[0] < 3]

assert nested == [(1, 10), (1, 20), (2, 10), (2, 20)]
assert pipeline == nested
assert nested_visits == pipeline_visits == 6
print("pairs:", pipeline)
print("candidate visits:", pipeline_visits)


pairs: [(1, 10), (1, 20), (2, 10), (2, 20)]
candidate visits: 6


### Try it: Option handling for a missing config key

Scala wraps a possibly-absent value in `Option` and chains with `map`/`flatMap` instead of null checks. Model a config lookup that returns `None` for a missing key.

Task: read `batch_size` from a config dict, parse it to an int, and fall back to 500 when any step is missing or invalid.

Hint: an `option_map(opt, f)` helper returns `None` when the input is `None` or when `f` raises.

Expected result: `{"batch_size": "250"}` gives 250; `{}` gives 500; `{"batch_size": "abc"}` gives 500.


In [3]:
def to_option(value):
    return None if value is None else ("some", value)


def option_map(opt, f):
    if opt is None:
        return None
    try:
        return to_option(f(opt[1]))
    except (ValueError, TypeError):
        return None


def option_get_or_else(opt, default):
    return default if opt is None else opt[1]


def parse_int(raw):
    return int(raw)


configs = [
    {"batch_size": "250"},
    {},
    {"batch_size": "abc"},
]

results = []
for cfg in configs:
    chained = option_map(to_option(cfg.get("batch_size")), parse_int)
    results.append(option_get_or_else(chained, 500))

assert results == [250, 500, 500]

# Option(null) in Scala is None, so an explicit None behaves like a missing key.
assert option_get_or_else(option_map(to_option(None), parse_int), 500) == 500
print("batch sizes:", results)


batch sizes: [250, 500, 500]


### Try it: reproduce pattern matching with dataclass records

A Scala `case class` gives structural equality, a `copy` method, and `unapply` for pattern matching. Python dataclasses give the first two, and structural `match` gives the third on Python 3.10+.

Task: build an `Order` dataclass, compare two equal orders, copy one with a new status, and route three event types with `match`.

Hint: `@dataclass(frozen=True)` makes the record immutable and hashable.

Expected result: two equal orders compare equal, the copy changes only `status`, and routing returns `insert`, `merge`, and `delete`.


In [4]:
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class Order:
    order_id: int
    customer_id: int
    amount: float
    status: str = "PENDING"


@dataclass(frozen=True)
class Created:
    order_id: int


@dataclass(frozen=True)
class Updated:
    order_id: int
    amount: float


@dataclass(frozen=True)
class Deleted:
    order_id: int


original = Order(1, 10, 19.99, "PENDING")
same = Order(1, 10, 19.99, "PENDING")
paid = replace(original, status="PAID")

assert original == same                 # structural equality, like a case class
assert paid != original
assert original.status == "PENDING"     # frozen: the original is unchanged
assert hash(original) == hash(same)


def route(event):
    match event:
        case Created():
            return "insert"
        case Updated(amount=a) if a > 100:
            return "merge"
        case Updated():
            return "merge"
        case Deleted():
            return "delete"
        case _:
            return "unknown"


assert route(Created(1)) == "insert"
assert route(Updated(1, 250.0)) == "merge"
assert route(Deleted(1)) == "delete"
print("routing:", route(Created(1)), route(Updated(1, 250.0)), route(Deleted(1)))


routing: insert merge delete


### Try it: fold a collection into a summary

Scala `foldLeft` carries an accumulator and can change the result type, turning a `List[Order]` into a `Map[String, BigDecimal]`. Model revenue by region with a single pass.

Task: fold a list of order dicts into a dict of region totals plus a count of paid orders, in one pass rather than one pass per region.

Hint: seed the accumulator with an empty structure and return it from each step.

Expected result: `us` revenue is 130.0, `eu` is 50.0, and the paid count is 3.


In [5]:
orders = [
    {"id": 1, "region": "us", "amount": 100.0, "status": "PAID"},
    {"id": 2, "region": "eu", "amount": 50.0, "status": "PAID"},
    {"id": 3, "region": "us", "amount": 30.0, "status": "PENDING"},
    {"id": 4, "region": "us", "amount": 30.0, "status": "PAID"},
]


def step(acc, order):
    by_region, paid_count = acc
    by_region = dict(by_region)
    if order["status"] == "PAID":
        by_region[order["region"]] = by_region.get(order["region"], 0.0) + order["amount"]
        paid_count += 1
    return by_region, paid_count


def fold_left(xs, seed, f):
    acc = seed
    for x in xs:
        acc = f(acc, x)
    return acc


by_region, paid_count = fold_left(orders, ({}, 0), step)

assert by_region == {"us": 130.0, "eu": 50.0}
assert paid_count == 3


def reduce_option(xs, f):
    if not xs:
        return None
    acc = xs[0]
    for x in xs[1:]:
        acc = f(acc, x)
    return acc


# reduce on an empty collection throws in Scala; reduceOption returns None.
assert reduce_option([], lambda a, b: a + b) is None
assert reduce_option([1, 2, 3], lambda a, b: a + b) == 6
print("revenue by region:", by_region)
print("paid orders:", paid_count)


revenue by region: {'us': 130.0, 'eu': 50.0}
paid orders: 3


### Try it: reproduce a for-comprehension as map/flatMap

Scala desugars `for { a <- xs; b <- ys } yield (a, b)` to `xs.flatMap(a => ys.map(b => (a, b)))`. Model a tiny option-like container with `map` and `flatMap`, then build the equivalent chain.

Task: implement `Some`/`None` with `map` and `flatMap`, and compute `a + b` from two lookups.

Hint: `flatMap` short-circuits on `None`.

Expected result: with both keys present the result is `Some(30)`; with any key missing it is `None`; both call styles agree.


In [6]:
class Opt:
    def __init__(self, value, present=True):
        self.value = value
        self.present = present

    def map(self, f):
        if not self.present:
            return self
        return Opt(f(self.value), True)

    def flat_map(self, f):
        if not self.present:
            return self
        return f(self.value)

    def __eq__(self, other):
        return (self.present, self.value) == (other.present, other.value)

    def __repr__(self):
        return "None" if not self.present else "Some({0})".format(self.value)


NONE = Opt(None, present=False)


def some(v):
    return Opt(v, True)


lookup = {1: 10, 2: 20}


def get_value(key):
    return some(lookup[key]) if key in lookup else NONE


def add(a):
    return lambda b: a + b


# for { a <- get_value(k1); b <- get_value(k2) } yield a + b
def for_style(k1, k2):
    return get_value(k1).flat_map(lambda a: get_value(k2).map(add(a)))


def explicit(k1, k2):
    return get_value(k1).flat_map(lambda a: get_value(k2).map(lambda b: a + b))


assert for_style(1, 2) == some(30)
assert for_style(1, 9) == NONE
assert for_style(9, 2) == NONE
assert for_style(1, 2) == explicit(1, 2)
print("present:", for_style(1, 2), "missing:", for_style(1, 9))


present: Some(30) missing: None


### Try it: why a captured variable must be serializable

Spark serializes a closure and everything it captures. A plain function defined at module level can be pickled; a lambda defined inline cannot, which is the Python analogue of `NotSerializableException`.

Task: show that a module-level function pickles, that a lambda does not, and that capturing a non-serializable object such as a lock or an open connection fails.

Hint: `pickle.dumps` raises for lambdas and live handles.

Expected result: the module-level function serializes, the lambda raises, and the captured handle raises with the failing type named.


In [7]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
import pickle
import threading


def double(x):
    return x * 2


blob = pickle.dumps(double)
assert isinstance(blob, bytes)

raised = None
try:
    pickle.dumps(lambda x: x * 2)
except Exception as exc:
    raised = type(exc).__name__
assert raised is not None
print("lambda serialization error:", raised)

handle = threading.Lock()   # a live handle, like a SparkSession
captured_error = None
try:
    pickle.dumps({"rows": handle})
except Exception as exc:
    captured_error = str(exc)
assert captured_error is not None
print("captured handle error:", captured_error.split(":")[0])

# A plain data record serializes, which is why case classes and encoders work.
record = {"order_id": 1, "amount": 19.99, "status": "PAID"}
assert pickle.loads(pickle.dumps(record)) == record
print("record round-trips:", pickle.loads(pickle.dumps(record)))


lambda serialization error: PicklingError
captured handle error: cannot pickle '_thread.lock' object
record round-trips: {'order_id': 1, 'amount': 19.99, 'status': 'PAID'}


### Try it: compare two collection strategies by operation counts

Appending to a Scala `List` copies the spine each time, so a loop that appends is O(n^2). Building with a `Vector` or a builder and converting once is O(n). Model both and count element copies.

Task: build 1,000 items by repeated append and by a single builder pass, counting copy operations in each.

Hint: the append strategy copies the current length on each step, giving n(n-1)/2 copies.

Expected result: append does 499,500 copies, the builder does 1,000, and both produce the same list.


In [8]:
n = 1000

append_copies = 0
appended = []
for i in range(n):
    append_copies += len(appended)   # concatenation copies the existing elements
    appended = appended + [i]

built = []
for i in range(n):
    built.append(i)                  # a builder grows in place
builder_copies = n
result = list(built)                 # one conversion, like .toVector

expected = n * (n - 1) // 2
assert append_copies == expected == 499500
assert builder_copies == 1000
assert appended == built == list(range(n))

# In Scala `i :: list` shares the tail, so cons is O(1); model the order only.
prepended = []
for i in range(n - 1, -1, -1):
    prepended.insert(0, i)
assert prepended == list(range(n))

print("append copies:", append_copies)
print("builder copies:", builder_copies)
print("speedup factor:", append_copies // builder_copies)


append copies: 499500
builder copies: 1000
speedup factor: 499
